# Framework vs No Framework

This file will go over some code we did in class, and then we will see how easy it is for that code to be replaced by the OpenAI Agents SDK

## Completion

### Plain

In [1]:
from dotenv import load_dotenv
from openai import Client
from time import time
load_dotenv()
client = Client()
response = client.responses.create(
    model="gpt-5-nano",
    instructions="respond only in emojis",
    input="Hello"
)

print(response.output_text)

👋😊


### Agents Framework

In [2]:
import asyncio
from agents import Agent, Runner

agent = Agent(
    name = "Test Agent",
    instructions="respons only in emojis",
    model="gpt-5-nano",
)
results = await Runner.run(agent, "Hello")
print(results.final_output)

👋😊💬❓


## Structured Output
### Plain

In [21]:
import json
structured_output = {
    "name": "ResearchResponse",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "summary": {
                "type": "string"
            },
            "key_entities": {
                "type": "array",
                "items": {
                    "type": "string"
                }
            },
            "confidence_score": {
                "type": "number"
            }
        },
        "required": [
            "summary",
            "key_entities",
            "confidence_score"
        ],
        "additionalProperties": False
    }
}

response = client.responses.create(
    model="gpt-5-nano",
    instructions="Analyze the input and provide a structured summary.",
    input="Tell me about the 2025 tech trends.",
    
    text={
        "format": {
            "type": "json_schema",
            **structured_output  # Unpacks name, strict, and schema
        }
    }
)

result_json = json.loads(response.output_text)

print(result_json["summary"])
print(result_json["key_entities"])

Forecast for 2025: AI adoption accelerates across sectors with generative AI and foundation models forming core platforms for products, services, and decision guidance. On‑device and edge AI enable private, low‑latency inference, while specialized AI chips and accelerators expand performance and efficiency. Cloud‑native, multi‑cloud architectures and AI‑first platforms continue to dominate software development, with automation and digital twins in manufacturing and logistics expanding the factory of the future. Networking upgrades from mature 5G and early 6G pilots enable real‑time, cross‑site collaboration and immersive remote work. Cybersecurity shifts toward zero‑trust models, post‑quantum cryptography, and privacy‑preserving ML and data protection. In parallel, quantum computing research makes incremental progress in optimization and cryptography applicability. Sustainability remains a priority, driving green software, energy‑efficient hardware, and climate‑tech solutions. Enterpri

### Agents Framework

In [22]:
from pydantic import BaseModel
# 1. Define the structure you want
class ResearchResponse(BaseModel):
    summary: str
    key_entities: list[str]
    confidence_score: float

# 2. Create the Agent with output_type
research_agent = Agent(
    name="Researcher",
    instructions="Analyze the input and provide a structured summary.",
    model = "gpt-5-nano",
    output_type=ResearchResponse,  # <--- This triggers Structured Output
)

# 3. Run the agent
result = await Runner.run(research_agent, input="Tell me about the 2025 tech trends.")

# The final_output is automatically parsed into your Pydantic object
print(f"Summary: {result.final_output.summary}")
print(f"Entities: {result.final_output.key_entities}")

Summary: 2025 tech trends are likely to center on AI becoming even more integrated, with enterprise copilots, code assistants, and machine-driven automation becoming standard. Expect hardware–software co-evolution: more on‑device and edge AI powered by specialized accelerators, while cloud-based AI services scale across industries. Automation and robotics expand in manufacturing, logistics, and services; digital twins and simulations optimize operations. Data infrastructure matures around multi‑cloud, data governance, privacy, and synthetic data to unlock responsible AI. Security and resilience rise in priority, with AI‑powered cybersecurity and zero‑trust models. Connectivity and immersive tech (5G/6G‑enabled) fuel new work and consumer experiences via AR/VR and remote collaboration. In healthcare, AI‑assisted diagnostics, imaging, and drug discovery accelerate. Climate tech and energy‑efficient computing gain traction in data centers and product design. Early progress in quantum comp

## Chatbot

### Plain

In [ ]:
import colorama
history = []
# if system_prompt:
#     history.append({'role': 'system', 'content': system_prompt})
client = Client()
while True:
    print(f"{colorama.Fore.GREEN}User{colorama.Fore.RESET}:", end=" ")
    prompt = input()
    print()
    if prompt.lower() == "stop":
        break
    history.append({'role': 'user', 'content': prompt})
    response = client.responses.create(
        model='gpt-5-nano',
        input=history,
        reasoning={"effort": "low"}
    )
    history.append({'role': 'assistant', 'content': response.output_text})
    print(f"{colorama.Fore.LIGHTBLUE_EX}agent{colorama.Fore.RESET}: {response.output_text}")
    print()

User: 

### Agents Framework

In [ ]:


# Initialize the agent
my_bot = Agent(
    name="NotebookBot",
    instructions="You are a helpful, concise assistant living in a Jupyter Notebook. Keep answers brief.",
    model="gpt-5-nano"
)

# This list holds your conversation history so the bot remembers context
chat_history = []

print("--- Chatbot Started (Type 'exit' to stop) ---")

while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit", "stop"]:
        print("Bot: catch you later!")
        break

    # Run the agent turn
    result = Runner.run_sync(
        my_bot,
        input=user_input,
        history=chat_history
    )

    # Save the updated history for the next turn
    chat_history = result.history

    # Print only the text response
    print(f"Bot: {result.final_output}")

--- Chatbot Started (Type 'exit' to stop) ---


## Reasoning

### Plain

In [5]:
client = Client()
response = client.responses.create(
    model="gpt-5-nano",
    instructions="respond only in emojis",
    input="Hello",
    reasoning={"effort": "minimal"}
)

print(response.output_text)

👋😊✨


### Agents Framework

In [8]:
from agents import ModelSettings
agent = Agent(
    name = "Test Agent",
    instructions="respons only in emojis",
    model="gpt-5-nano",
    model_settings=ModelSettings(reasoning={"effort": "minimal"})
)
results = await Runner.run(agent, "Hello")
print(results.final_output)

👋😊


# Just Framework Implementations

## Function Tools

In [9]:
from agents import function_tool

@function_tool
def history_fun_fact() -> str:
    """Return a short history fact."""
    return "Sharks are older than trees."


agent = Agent(
    name="History tutor",
    instructions="Answer history questions clearly. Use history_fun_fact when it helps.",
    model='gpt-5-nano',
    tools=[history_fun_fact],
)

result = await Runner.run(
    agent,
    "Tell me something surprising about ancient life on Earth.",
)
print(result.final_output)

Sharks are older than trees. The earliest sharks appeared around 400 million years ago, while the first trees didn’t show up until about 350–380 million years ago.


## Agents as Tools

In [ ]:
from agents import AgentHooks

class MyAgentHooks(AgentHooks):
    # Correct 2026 Signature: (self, context, agent, tool)
    async def on_tool_start(self, context, agent, tool):
        # 1. 'context' is a ToolContext object
        # 2. 'agent' is the Research Assistant
        # 3. 'tool' is the Summarizer (as a tool)
        
        print(f"--- 🚀 {agent.name} is calling: {tool.name} ---")
        
        # Accessing arguments correctly
        try:
            # In the latest SDK, tool_call is directly on the context
            args = context.tool_call.function.arguments
            print(f"--- 📦 Arguments: {args}")
        except AttributeError:
            # Fallback for some sub-versions/handoff types
            print("--- 📦 Could not parse arguments directly from context.")

    async def on_tool_end(self, context, agent, tool, result):
        print(f"--- ✅ Tool {tool.name} finished ---")
        # 'result' is the actual summary returned by the summarizer
        print(f"--- 📄 Summary result: {result[:50]}...")

my_hooks = MyAgentHooks()

summarizer = Agent(
    name="Summarizer",
    instructions="Generate a concise summary of the supplied text.",
)

main_agent = Agent(
    name="Research assistant",
    tools=[
        summarizer.as_tool(
            tool_name="summarize_text",
            tool_description="Generate a concise summary of the supplied text.",
        )
    ],
    hooks= my_hooks
)
result = await Runner.run(
    main_agent,
    """please summarize the following text:
        23 And I said unto him: Lord, the Gentiles will mock at these things, because of our weakness in writing; for Lord thou hast made us mighty in word by faith, but thou hast not made us mighty in writing; for thou hast made all this people that they could speak much, because of the Holy Ghost which thou hast given them;
        24 And thou hast made us that we could write but little, because of the awkwardness of our hands. Behold, thou hast not made us mighty in writing like unto the brother of Jared, for thou madest him that the things which he wrote were mighty even as thou art, unto the overpowering of man to read them.
        25 Thou hast also made our words powerful and great, even that we cannot write them; wherefore, when we write we behold our weakness, and stumble because of the placing of our words; and I fear lest the Gentiles shall mock at our words.
        26 And when I had said this, the Lord spake unto me, saying: Fools mock, but they shall mourn; and my grace is sufficient for the meek, that they shall take no advantage of your weakness;
        27 And if men come unto me I will show unto them their weakness. I give unto men weakness that they may be humble; and my grace is sufficient for all men that humble themselves before me; for if they humble themselves before me, and have faith in me, then will I make weak things become strong unto them.
        28 Behold, I will show unto the Gentiles their weakness, and I will show unto them that faith, hope and charity bringeth unto me—the fountain of all righteousness.""",
)
print(result.final_output)

--- 🚀 Research assistant is calling: summarize_text ---
--- 📦 Could not parse arguments directly from context.
--- ✅ Tool summarize_text finished ---
--- 📄 Summary result: The speaker expresses concern to the Lord that the...
The speaker worries that their weakness in writing will cause others (the Gentiles) to mock, even though their spoken words are powerful by faith. The Lord answers that those who mock will regret it, and that His grace helps the humble. God says weaknesses are given to help people be humble, and if they have faith and humble themselves, He will turn their weaknesses into strengths. Ultimately, God will show that faith, hope, and charity lead to righteousness.
